In [1]:
"""
Fine-tuning script para rodar no Kaggle
======================================
Como usar:
1. Faça upload do seu modelo (.pt) e do txt de fine-tuning para o Kaggle
2. Ajuste os caminhos em CONFIGURAÇÃO abaixo
3. Rode o notebook

Estrutura esperada no Kaggle:
/kaggle/input/seu-dataset/modelo_loss_2.2.pt
/kaggle/input/seu-dataset/finetuning.txt
"""

import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict
import time
import os

# ============================================================
# ⚙️  CONFIGURAÇÃO — EDITE AQUI
# ============================================================

MODEL_PATH    = "/kaggle/input/models/brenocamposribeiro/modelo-final/pytorch/default/1/modelo_finetuned_best.pt"
FINETUNE_TXT  = "/kaggle/input/datasets/brenocamposribeiro/novojavascript/Ol o que  javascript.txt"
OUTPUT_PATH   = "/kaggle/working/modelo_finetuned.pt"

# Hiperparâmetros de fine-tuning
EPOCHS        = 3       # poucas épocas evitam catastrophic forgetting
BATCH_SIZE    = 16
BLOCK_SIZE    = 256
LR            = 1e-4    # lr baixo é importante no fine-tuning
WARMUP_STEPS  = 100
GRAD_CLIP     = 1.0
SAVE_EVERY    = 1       # salvar checkpoint a cada N épocas

# Se True, congela as camadas de embedding (treina só o transformer)
FREEZE_EMBEDDINGS = False

# ⚠️ IMPORTANTE: mude para True se seu finetuning.txt usa formato usuário/assistente
IS_DIALOGUE = True

# ============================================================
# 🔧 CLASSES (copie as mesmas do seu script original)
# ============================================================

class SimpleWordTokenizer:
    def __init__(self, min_freq: int = 2, max_vocab: int = 30000):
        self.min_freq = min_freq
        self.max_vocab = max_vocab
        self.stoi: Dict[str, int] = {}
        self.itos: Dict[int, str] = {}
        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.nl_token  = "<nl>"
        self._re_tok   = re.compile(r"\n|\w+|[^\w\s]", flags=re.UNICODE)

    def _tokenize(self, text: str) -> List[str]:
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        raw = self._re_tok.findall(text.lower())
        return [self.nl_token if t == "\n" else t for t in raw]

    def build_vocab(self, text: str):
        tokens = self._tokenize(text)
        freq: Dict[str, int] = {}
        for t in tokens:
            freq[t] = freq.get(t, 0) + 1
        reserved = {self.pad_token, self.unk_token, self.nl_token}
        sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)
        sorted_tokens = [t for t, c in sorted_tokens if c >= self.min_freq and t not in reserved]
        sorted_tokens = sorted_tokens[: self.max_vocab - 3]
        self.stoi = {self.pad_token: 0, self.unk_token: 1, self.nl_token: 2}
        for i, tok in enumerate(sorted_tokens, start=3):
            self.stoi[tok] = i
        self.itos = {i: s for s, i in self.stoi.items()}

    def encode(self, text: str) -> List[int]:
        tokens = self._tokenize(text)
        unk_id = self.stoi[self.unk_token]
        return [self.stoi.get(t, unk_id) for t in tokens]

    def decode(self, ids: List[int]) -> str:
        tokens = [self.itos.get(i, self.unk_token) for i in ids]
        return self._detokenize(tokens)

    def _detokenize(self, tokens: List[str]) -> str:
        if not tokens:
            return ""
        no_space_before = {",", ".", "!", "?", ";", ":", ")", "]", "}"}
        no_space_after  = {"(", "[", "{"}
        out = []
        for i, tok in enumerate(tokens):
            if tok == self.nl_token:
                out.append("\n"); continue
            if i == 0:
                out.append(tok); continue
            prev = tokens[i - 1]
            add_space = tok not in no_space_before and prev not in no_space_after and prev != self.nl_token
            if add_space:
                out.append(" ")
            out.append(tok)
        return "".join(out)

    @property
    def vocab_size(self) -> int:
        return len(self.stoi)


class GPTConfig:
    def __init__(self, vocab_size, d_model=256, n_heads=4,
                 n_layers=4, d_ff=1024, max_seq_len=128, dropout=0.1):
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.n_heads     = n_heads
        self.n_layers    = n_layers
        self.d_ff        = d_ff
        self.max_seq_len = max_seq_len
        self.dropout     = dropout


class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb   = nn.Embedding(config.max_seq_len, config.d_model)
        decoder_layer  = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.d_ff, dropout=config.dropout,
            activation="gelu", batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=config.n_layers)
        self.ln_f = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size)

    def forward(self, idx: torch.Tensor):
        B, T = idx.shape
        device = idx.device
        positions = torch.arange(T, device=device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(positions)
        mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
        x = self.transformer(x, mask=mask)
        x = self.ln_f(x)
        return self.head(x)


class TextDataset(Dataset):
    def __init__(self, data: List[int], block_size: int):
        self.data       = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx      : idx + self.block_size],     dtype=torch.long)
        y = torch.tensor(self.data[idx + 1  : idx + self.block_size + 1], dtype=torch.long)
        return x, y


# ============================================================
# 📦 CARREGAR MODELO PRÉ-TREINADO
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\n📂 Carregando modelo de: {MODEL_PATH}")
checkpoint = torch.load(MODEL_PATH, map_location=device)

cfg   = GPTConfig(**checkpoint["config"])
model = GPTModel(cfg).to(device)
model.load_state_dict(checkpoint["model_state"])

# Reconstruir tokenizer (usa o vocabulário já treinado)
tokenizer = SimpleWordTokenizer()
tokenizer.stoi = checkpoint["tokenizer"]
tokenizer.itos = {i: s for s, i in tokenizer.stoi.items()}

print(f"✅ Modelo carregado  |  Vocab: {tokenizer.vocab_size}  |  Loss anterior: {checkpoint.get('loss', 'N/A')}")

# ============================================================
# 📄 CARREGAR E TOKENIZAR O TXT DE FINE-TUNING
# ============================================================

print(f"\n📖 Lendo arquivo de fine-tuning: {FINETUNE_TXT}")
with open(FINETUNE_TXT, "r", encoding="utf-8") as f:
    ft_text = f.read()

print(f"   Caracteres: {len(ft_text):,}")
print(f"   Primeiros 200 chars: {ft_text[:200]!r}")

# ⚠️ Aviso de maiúsculas — o tokenizer faz .lower() em tudo
upper_count = sum(1 for c in ft_text if c.isupper())
if upper_count > 0:
    print(f"   ⚠️  {upper_count} letras maiúsculas detectadas! O tokenizer converte tudo para minúsculas.")
    print(f"       Isso é esperado, mas verifique se não há siglas importantes que precisem de atenção.")


# Tokeniza com o vocab existente (sem re-treinar o vocab!)
ft_ids = tokenizer.encode(ft_text)
print(f"   Tokens: {len(ft_ids):,}")

# Calcula quantos tokens foram mapeados como <unk>
unk_id    = tokenizer.stoi[tokenizer.unk_token]
unk_count = ft_ids.count(unk_id)
unk_pct   = unk_count / len(ft_ids) * 100
print(f"   Tokens desconhecidos (<unk>): {unk_count:,} ({unk_pct:.1f}%)")
if unk_pct > 30:
    print("   ⚠️  Muitos tokens desconhecidos! O texto pode ser muito diferente do pré-treino.")

# Split treino/validação (90/10)
split     = int(0.9 * len(ft_ids))
train_ids = ft_ids[:split]
val_ids   = ft_ids[split:]

train_dataset = TextDataset(train_ids, BLOCK_SIZE)
val_dataset   = TextDataset(val_ids,   BLOCK_SIZE)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2, pin_memory=True)

print(f"\n📊 Split treino/val: {len(train_ids):,} / {len(val_ids):,} tokens")
print(f"   Batches por época: {len(train_loader):,}")

# ============================================================
# 🔒 CONGELAR EMBEDDINGS (OPCIONAL)
# ============================================================

if FREEZE_EMBEDDINGS:
    model.token_emb.weight.requires_grad_(False)
    model.pos_emb.weight.requires_grad_(False)
    print("🔒 Embeddings congelados (só transformer e head serão treinados)")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🔢 Parâmetros treináveis: {trainable:,} / {total:,}")

# ============================================================
# ⚡ OPTIMIZER + SCHEDULER
# ============================================================

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

# Linear warmup + cosine decay
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(total_steps - WARMUP_STEPS, 1)
    return max(0.1, 0.5 * (1.0 + torch.cos(torch.tensor(3.14159 * progress)).item()))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ============================================================
# 🏋️  LOOP DE FINE-TUNING
# ============================================================

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x)
                loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
    return total_loss / len(loader)


print("\n" + "="*60)
print("🚀 INICIANDO FINE-TUNING")
print("="*60)

best_val_loss = float("inf")
global_step   = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss  = 0
    t0          = time.time()

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss  += loss.item()
        global_step += 1

        # Log a cada 50 batches
        if (batch_idx + 1) % 50 == 0:
            avg   = epoch_loss / (batch_idx + 1)
            lr_now = scheduler.get_last_lr()[0]
            elapsed = time.time() - t0
            print(f"  Época {epoch} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {avg:.4f} | LR: {lr_now:.2e} | {elapsed:.0f}s")

    # Validação ao final da época
    train_loss = epoch_loss / len(train_loader)
    val_loss   = evaluate(model, val_loader, device)
    elapsed    = time.time() - t0

    print(f"\n📈 Época {epoch}/{EPOCHS}  "
          f"| Train Loss: {train_loss:.4f}  "
          f"| Val Loss: {val_loss:.4f}  "
          f"| Tempo: {elapsed:.0f}s")

    # Salvar melhor modelo
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path     = OUTPUT_PATH.replace(".pt", "_best.pt")
        torch.save({
            "epoch":        epoch,
            "loss":         val_loss,
            "model_state":  model.state_dict(),
            "config":       checkpoint["config"],
            "tokenizer":    tokenizer.stoi,
            "is_dialogue":  IS_DIALOGUE,
        }, best_path)
        print(f"  💾 Melhor modelo salvo! Val Loss: {val_loss:.4f} → {best_path}")

    # Salvar checkpoint por época
    if epoch % SAVE_EVERY == 0:
        ckpt_path = OUTPUT_PATH.replace(".pt", f"_ep{epoch}.pt")
        torch.save({
            "epoch":        epoch,
            "loss":         val_loss,
            "model_state":  model.state_dict(),
            "config":       checkpoint["config"],
            "tokenizer":    tokenizer.stoi,
            "is_dialogue":  IS_DIALOGUE,
        }, ckpt_path)
        print(f"  💾 Checkpoint salvo: {ckpt_path}")

# Salvar modelo final
torch.save({
    "epoch":        EPOCHS,
    "loss":         val_loss,
    "model_state":  model.state_dict(),
    "config":       checkpoint["config"],
    "tokenizer":    tokenizer.stoi,
    "is_dialogue":  IS_DIALOGUE,
}, OUTPUT_PATH)

print("\n" + "="*60)
print(f"✅ FINE-TUNING CONCLUÍDO!")
print(f"   Val Loss final:  {val_loss:.4f}")
print(f"   Melhor Val Loss: {best_val_loss:.4f}")
print(f"   Modelo final:    {OUTPUT_PATH}")
print(f"   Melhor modelo:   {OUTPUT_PATH.replace('.pt', '_best.pt')}")
print("="*60)

# ============================================================
# 🧪 TESTE RÁPIDO
# ============================================================

print("\n🧪 Teste rápido com o modelo fine-tunado:")

model.eval()

# Para diálogo, o prompt deve terminar com "\nassistente:" para o modelo completar
if IS_DIALOGUE:
    prompts_teste = [
        "usuário: olá\nassistente:",
        "usuário: quem é você?\nassistente:",
        "usuário: aristóteles\nassistente:",
        "usuário: o que é javascript?\nassistente:",
    ]
else:
    prompts_teste = [
        "filosofia",
        "inteligência artificial",
        "brasil",
    ]

for prompt in prompts_teste:
    ids = tokenizer.encode(prompt)
    if not ids:
        ids = [tokenizer.stoi[tokenizer.unk_token]]
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(40):
            idx_cond = idx[:, -cfg.max_seq_len:]
            logits   = model(idx_cond)
            logits   = logits[:, -1, :] / 0.7
            # top-k 40
            v, i     = torch.topk(logits, 40)
            probs    = F.softmax(v, dim=-1)
            next_tok = i.gather(-1, torch.multinomial(probs, 1))
            idx      = torch.cat([idx, next_tok], dim=1)

    print(f"\n  Prompt: '{prompt}'")
    print(f"  Saída:  {tokenizer.decode(idx[0].tolist())[:200]}")
    print("  " + "-"*50)


🖥️  Dispositivo: cuda
🎮 GPU: Tesla T4
💾 Memória: 15.6 GB

📂 Carregando modelo de: /kaggle/input/models/brenocamposribeiro/modelo-final/pytorch/default/1/modelo_finetuned_best.pt
✅ Modelo carregado  |  Vocab: 50000  |  Loss anterior: 7.2160217508356626

📖 Lendo arquivo de fine-tuning: /kaggle/input/datasets/brenocamposribeiro/novojavascript/Ol o que  javascript.txt
   Caracteres: 6,417
   Primeiros 200 chars: 'usuário: Olá, o que é javascript?\nassistente: JavaScript é uma linguagem de programação de alto nível, interpretada e dinâmica, essencial para criar sites interativos e funcionais. Ao lado de HTML e C'
   ⚠️  122 letras maiúsculas detectadas! O tokenizer converte tudo para minúsculas.
       Isso é esperado, mas verifique se não há siglas importantes que precisem de atenção.
   Tokens: 1,901
   Tokens desconhecidos (<unk>): 242 (12.7%)

📊 Split treino/val: 1,710 / 191 tokens
   Batches por época: 90
🔢 Parâmetros treináveis: 28,875,088 / 28,875,088

🚀 INICIANDO FINE-TUNING


/tmp/ipykernel_57/2477431942.py:265: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_57/2477431942.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


  Época 1 | Batch 50/90 | Loss: 5.3028 | LR: 5.00e-05 | 4s


ValueError: __len__() should return >= 0